# Form Engine — LoRA fine-tune of Qwen2.5-Coder-7B-Instruct

Run this on a Colab **T4 (free)** or **A100 (pro)** runtime.

**Inputs**
- `dataset.jsonl` — produced by `backend/training/generate_dataset.py`. Upload it to the Colab session at `/content/dataset.jsonl` (or mount Drive).

**Outputs**
- `form-engine-qwen.gguf` — quantised model ready to register with `ollama create`.
- `lora-adapter/` — the raw LoRA weights if you want to apply them differently later.

**End-to-end runtime** on a free T4 with ~80 samples and 3 epochs: ~15 min.

## 1. Install Unsloth + dependencies

In [ ]:
%%capture
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install pyyaml

## 2. Load Qwen2.5-Coder-7B in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 4096   # most form YAMLs + system fit comfortably under this
BASE_MODEL  = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,    # auto: bfloat16 on Ampere+, float16 on T4
    load_in_4bit    = True,
)

## 3. Attach LoRA adapters

Targeting the standard QLoRA modules. `r=16, alpha=16` is a safe default for tiny
datasets (<200 samples) — larger ranks overfit fast.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    random_state   = 42,
)

## 4. Load + format the dataset

Each JSONL line becomes a 2-turn chat:
- **user** — the natural-language description
- **assistant** — the YAML wrapped in a ```yaml fenced block

We use Qwen's ChatML template via `tokenizer.apply_chat_template`.

In [ ]:
import json, pathlib
from datasets import Dataset

DATASET_PATH = pathlib.Path("/content/dataset.jsonl")

SYSTEM_INSTRUCTION = (
    "You are a Form Engine YAML generator. Given a user's description of a form, "
    "produce a complete, schema-valid YAML manifest wrapped in a single ```yaml "
    "fenced block and nothing else."
)

def _load() -> Dataset:
    records = []
    with DATASET_PATH.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            messages = [
                {"role": "system",    "content": SYSTEM_INSTRUCTION},
                {"role": "user",      "content": rec["description"]},
                {"role": "assistant", "content": f"```yaml\n{rec['yaml']}\n```"},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
            )
            records.append({"text": text})
    return Dataset.from_list(records)

ds = _load()
print(f"Loaded {len(ds)} training examples")
print("\n--- sample ---")
print(ds[0]["text"][:1200])

## 5. Train

Tuned for ~80-150 samples on a T4. If you have more data or a bigger GPU, raise
`per_device_train_batch_size` and lower `gradient_accumulation_steps`
proportionally.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,    # effective batch = 8
        warmup_ratio                = 0.05,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 42,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## 6. Smoke-test: generate a manifest and validate it

In [ ]:
import re, yaml as _yaml

FastLanguageModel.for_inference(model)

def generate(description: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user",   "content": description},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")
    out = model.generate(
        input_ids      = inputs,
        max_new_tokens = 1500,
        do_sample      = False,
        temperature    = 0.0,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

test_prompts = [
    "A simple contact form: name, email, subject dropdown, message textarea.",
    "A wizard for booking a dental appointment with patient details, symptoms, preferred date and a follow-up question that only shows for new patients.",
]
for prompt in test_prompts:
    print(f"\n=== {prompt}\n")
    reply = generate(prompt)
    print(reply[:1500])
    m = re.search(r"```yaml\s*([\s\S]*?)```", reply)
    if m:
        try:
            _yaml.safe_load(m.group(1))
            print("\n  ✅ YAML parses")
        except _yaml.YAMLError as e:
            print(f"\n  ❌ YAML invalid: {e}")
    else:
        print("\n  ❌ no yaml block in reply")

## 7. Save LoRA adapter (raw, ~150MB)

In [ ]:
model.save_pretrained("lora-adapter")
tokenizer.save_pretrained("lora-adapter")
!ls -lh lora-adapter

## 8. Export to GGUF for Ollama

`save_pretrained_gguf` merges the LoRA into the base weights and quantises in
one step. `q4_k_m` is the recommended balance — ~4.7GB and visually identical
quality to fp16 for this task. Use `q5_k_m` if you have RAM to spare.

**This step takes 10-15 min** because it re-downloads the full-precision base
weights to merge against.

In [ ]:
model.save_pretrained_gguf(
    "form-engine-qwen",
    tokenizer,
    quantization_method = "q4_k_m",
)
!ls -lh form-engine-qwen*

## 9. Download & register with Ollama

Download `form-engine-qwen-unsloth.Q4_K_M.gguf` from the Colab file browser,
then on the machine running Ollama:

```bash
# Create a Modelfile pointing at the gguf
cat > Modelfile <<'EOF'
FROM ./form-engine-qwen-unsloth.Q4_K_M.gguf
TEMPLATE """<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""
PARAMETER temperature 0.2
PARAMETER stop "<|im_end|>"
EOF

ollama create form-engine-qwen -f Modelfile
```

Then in the Form Engine backend set:

```bash
export LLM_PROVIDER=ollama
export OLLAMA_MODEL=form-engine-qwen
```

and restart the API. The `/create` mode in chat will now use your fine-tuned model.